<a href="https://colab.research.google.com/github/roughhawkbit/digi-inno-road-prod/blob/main/analysis/1_0_dataset_characterisation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook takes an initial look at the data, calculating the completeness & uniqueness of each column and checking that the two data sheets are aligned.

# Setup

In [ ]:
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    repo_path = '/content/drive/MyDrive/digi-inno-road-prod'
    if os.path.isdir(repo_path):
      cwd = os.getcwd()
      os.chdir(repo_path)
      !git pull
      os.chdir(cwd)
    else:
      !git clone https://github.com/roughhawkbit/digi-inno-road-prod.git /content/drive/MyDrive/digi-inno-road-prod
      print('Repository cloned into your Google Drive. It is strongly recommended that you copy the credentials.json, sheet.json, and token.json files into the secrets folder before proceeding.')
    sys.path.insert(0, repo_path)
    IN_COLAB = True
except ImportError:
    repo_path = os.path.abspath(os.path.join('../src'))
    IN_COLAB = False

if not repo_path in sys.path:
    sys.path.insert(0, repo_path)

In [ ]:
if IN_COLAB:
  output_path = os.path.join(repo_path, 'analysis', 'outputs')
else:
  output_path = os.path.join('.', 'outputs')
output_path = os.path.abspath(output_path)

# Import packages & data

Conda packages

In [ ]:
import pandas

Project sourcecode packages

In [ ]:
from innoprod.sheet_tools import get_sheet_dfs
from innoprod.wrangling.wrangling_tools import characterise_df_columnwise, is_non_empty
from innoprod.wrangling.msyh_data_sharing import wrangle_roadmaps, wrangle_grants

Data

In [ ]:
data = get_sheet_dfs()

INCLUDE_NO_GRANTS_FIRMS = True
if INCLUDE_NO_GRANTS_FIRMS:
  roadmaps_df = pandas.concat([data['Roadmaps'], data['RoadmapsWithoutGrants']])
else:
  roadmaps_df = data['Roadmaps']
roadmaps_df = wrangle_roadmaps(roadmaps_df)

grants_df = wrangle_grants(data['Grants'])

# Basic Data Characertisation

In [ ]:
roadmaps_characterisation = characterise_df_columnwise(roadmaps_df)
roadmaps_characterisation.to_csv(os.path.join(output_path, 'characterisation_roadmaps.csv'))

In [ ]:

grants_characterisation = characterise_df_columnwise(grants_df)
grants_characterisation.to_csv(os.path.join(output_path, 'characterisation_grants.csv'))

# Firm size

## Part-Time employees

In [ ]:
col = 'Number of PT employees'
mask = (roadmaps_df[col] == '') == (roadmaps_df[col].isna())
roadmaps_df[mask][['Number of FT employees', col]]

## Org Size

In [ ]:
col = 'Org Size by Number of FTE (calc)'
roadmaps_df.groupby(col).size()

## Zeros in FTE Employees

For many firms, the number of FTE employees is recorded as zero or is missing.

In [ ]:
zero_fte_mask = (roadmaps_df['Number of FTE Employees (calc)'] == 0).fillna(False)
int(sum(zero_fte_mask))

In [ ]:
nan_fte_mask = roadmaps_df['Number of FTE Employees (calc)'].isna()
sum(nan_fte_mask)

# Willing to be approached for case study

In [ ]:
willing_mask = (roadmaps_df['Willing to be approached for case study?'] == 'Yes')
sum(willing_mask)

# Cross-referencing Roadmaps and Grants

The `Client ID` values in the Grants dataset is a subset of those in the Roadmaps dataset:

In [ ]:
set(grants_df['Client ID']).issubset(set(roadmaps_df['Client ID']))

And the set of `Client ID`s matches perfectly when we look only at those firms in the `Roadmaps&SDRs` sheet that submitted at least one Grant Application Form (GAF).

In [ ]:
set(grants_df['Client ID']) == set(roadmaps_df[roadmaps_df['Number of GAFs'] > 0]['Client ID'])

The total number of grants matches the sum of the `Number of GAFs` in the `Roadmaps&SDRs` dataset

In [ ]:
bool(sum(roadmaps_df['Number of GAFs']) == grants_df['Grant ID'].count())